# 03 Model
Train LR, RF, and XGB without leaking `G1`, `G2`, or `G3` into model features.

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine, text
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, RocCurveDisplay, accuracy_score

engine = create_engine('sqlite:///../data/students.db')
df = pd.read_sql(text('SELECT * FROM students'), engine)
y = df['pass']
X = df.drop(columns=['G1','G2','G3','pass']).select_dtypes(include=[np.number])
joblib.dump(list(X.columns), '../src/feature_cols.pkl')
feature_cols = joblib.load('../src/feature_cols.pkl')
X = X[feature_cols]
X.shape

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
models = {
    'LR': (LogisticRegression(max_iter=1000, random_state=42), X_train_scaled, X_test_scaled),
    'RF': (RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42), X_train, X_test),
    'XGB': (XGBClassifier(eval_metric='logloss', random_state=42, verbosity=0), X_train, X_test),
}
metrics = []
for name, (model, train_x, test_x) in models.items():
    model.fit(train_x, y_train)
    pred = model.predict(test_x)
    prob = model.predict_proba(test_x)[:, 1]
    print(name)
    print(classification_report(y_test, pred, target_names=['Fail','Pass']))
    cm = confusion_matrix(y_test, pred)
    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Fail','Pass'], yticklabels=['Fail','Pass'])
    plt.title(f'{name} Confusion Matrix')
    plt.tight_layout()
    plt.show()
    plt.close()
    report = classification_report(y_test, pred, target_names=['Fail','Pass'], output_dict=True)
    metrics.append({'Model': name, 'Accuracy': accuracy_score(y_test, pred), 'ROC-AUC': roc_auc_score(y_test, prob), 'Fail Recall': report['Fail']['recall']})

In [ ]:
plt.figure(figsize=(8,6))
for name, (model, _, test_x) in models.items():
    RocCurveDisplay.from_estimator(model, test_x, y_test, name=name)
plt.title('ROC Curves')
plt.tight_layout()
plt.show()
plt.close()

In [ ]:
metrics_df = pd.DataFrame(metrics)
labels = metrics_df['Model']
x = np.arange(len(labels))
width = 0.35
fig, ax = plt.subplots(figsize=(9,5))
b1 = ax.bar(x - width/2, metrics_df['Accuracy']*100, width, label='Accuracy %')
b2 = ax.bar(x + width/2, metrics_df['ROC-AUC']*100, width, label='ROC-AUC %')
ax.axhline(80, color='red', linestyle='--', label='80% reference')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylim(0, 105)
ax.legend()
for bars in (b1, b2):
    for bar in bars:
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1, f'{bar.get_height():.1f}%', ha='center')
plt.tight_layout()
plt.savefig('../data/model_comparison.png', dpi=180)
plt.show()
plt.close()

In [ ]:
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 5, 10, 15],
    'min_samples_split': [2, 5, 10],
    'class_weight': ['balanced']
}
grid = GridSearchCV(RandomForestClassifier(random_state=42), param_grid=param_grid, cv=5, scoring='recall', n_jobs=-1)
grid.fit(X_train, y_train)
print('Best params:', grid.best_params_)
print(f'Best CV recall score: {grid.best_score_:.3f}')

In [ ]:
best_model = grid.best_estimator_
cv_scores = cross_val_score(best_model, X_train, y_train, cv=5, scoring='recall')
print(f'Mean recall: {cv_scores.mean():.3f}')
print(f'Std recall: {cv_scores.std():.3f}')

In [ ]:
rf_default = models['RF'][0]
default_pred = rf_default.predict(X_test)
default_prob = rf_default.predict_proba(X_test)[:, 1]
tuned_pred = best_model.predict(X_test)
tuned_prob = best_model.predict_proba(X_test)[:, 1]
print(f'Default RF accuracy: {accuracy_score(y_test, default_pred):.3f}')
print(f'Default RF ROC-AUC: {roc_auc_score(y_test, default_prob):.3f}')
print(f'Tuned RF accuracy: {accuracy_score(y_test, tuned_pred):.3f}')
print(f'Tuned RF ROC-AUC: {roc_auc_score(y_test, tuned_prob):.3f}')

In [ ]:
joblib.dump(best_model, '../src/model.pkl')
joblib.dump(scaler, '../src/scaler.pkl')
joblib.dump(list(X.columns), '../src/feature_cols.pkl')